<a href="https://colab.research.google.com/github/novikovamaria137-png/mtuci-llm-course/blob/main/lesson-2.3/practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# Практика 2.3. Структурированный вывод

**Модуль 2 · Урок 3 · 110 минут**

До сих пор ответ модели читал человек. Сегодня его будет читать программа — а значит, ответ должен иметь заданную форму, и эту форму надо проверять.

---

### Что вы сделаете

| Шаг | Что делаем | Время | Нужен ключ |
|---|---|---|---|
| 0 | Восстановим каркас уроков 2.1–2.2 | 10 мин | нет |
| 1 | Напишем модуль разбора структурированных ответов | 25 мин | нет |
| 2 | Прогоним разбор на десяти видах «грязного» ответа | 20 мин | нет |
| 3 | Добавим проверку формы и соберём все претензии сразу | 15 мин | нет |
| 4 | Соберём цикл починки с ограничением попыток | 20 мин | нет |
| 5 | Разберём, что гарантирует схема, а что нет | 10 мин | нет |
| 6 | Запросим структурированный ответ у модели | 10 мин | да |

> **Ключ нужен только на последнем шаге.** Всё остальное проверяется на записанных ответах — и это не упрощение, а правильный способ тестировать разбор: настоящая модель каждый раз отвечает по-разному, и на ней нельзя проверить обработку конкретного дефекта.

---
## Шаг 0. Каркас

Три модуля из прошлых уроков — без изменений. Colab стирает файлы между сеансами, поэтому собираем заново.

*Статус ячейки: проверено запуском.*

In [ ]:
REQUIREMENTS = ["openai==2.51.0", "python-dotenv==1.2.2"]

import importlib.util, subprocess, sys
from pathlib import Path

def ensure(spec):
    name = spec.split("==")[0].replace("-", "_")
    if importlib.util.find_spec(name) is None:
        print(f"  устанавливаю {spec} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=False)
    else:
        print(f"  {name}: уже установлен")

print("Зависимости:")
for spec in REQUIREMENTS:
    ensure(spec)

ROOT = Path("llm-project")
(ROOT / "llmcourse").mkdir(parents=True, exist_ok=True)
(ROOT / "llmcourse" / "__init__.py").write_text("", encoding="utf-8")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

print("\nПроект:", ROOT.resolve())

In [ ]:
%%writefile llm-project/llmcourse/config.py
"""Единая точка настройки. Урок 2.1.

Ключи НИКОГДА не пишутся в коде. Порядок поиска:
  1. Colab Secrets  (значок ключа слева в Colab)
  2. переменные окружения
  3. файл .env рядом с проектом
Если ключа нет — включается автономный режим на заглушке.
"""
import os
from pathlib import Path

ENV_KEYS = ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL")


def _from_colab(name):
    try:
        from google.colab import userdata          # есть только в Colab
        return userdata.get(name)
    except Exception:
        return None


def _from_dotenv(name, path=".env"):
    p = Path(path)
    if not p.exists():
        return None
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        if k.strip() == name:
            return v.strip().strip('"').strip("'")
    return None


def get(name, default=None):
    """Достаёт значение из Colab Secrets, окружения или .env."""
    return _from_colab(name) or os.environ.get(name) or _from_dotenv(name) or default


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def settings():
    """Возвращает конфигурацию и признак автономного режима."""
    cfg = {k: get(k) for k in ENV_KEYS}
    cfg["OFFLINE"] = not bool(cfg["LLM_API_KEY"])
    cfg["MODEL"] = cfg["LLM_MODEL"] or "demo-model"
    return cfg


def describe():
    """Человекочитаемый отчёт об окружении."""
    cfg = settings()
    where = "Google Colab" if in_colab() else "локальная среда"
    return "\n".join([
        f"Среда:            {where}",
        f"Модель:           {cfg['MODEL']}",
        f"Базовый адрес:    {cfg['LLM_BASE_URL'] or 'не задан'}",
        f"Ключ:             {'найден' if not cfg['OFFLINE'] else 'НЕ найден'}",
        f"Режим:            {'автономный (заглушка)' if cfg['OFFLINE'] else 'обращение к API'}",
    ])


In [ ]:
%%writefile llm-project/llmcourse/client.py
"""Клиент для работы с языковой моделью. Уроки 2.1–2.2.

Написан на OpenAI-совместимый интерфейс: работает с российскими API
и с локальными рантаймами. Смена поставщика — правка .env, не кода.
Без ключа работает в автономном режиме на заглушке.
"""
import time, random, hashlib
from dataclasses import dataclass
from . import config


@dataclass
class Usage:
    """Накопительный счётчик расхода."""
    calls: int = 0
    tokens_in: int = 0
    tokens_out: int = 0
    price_in: float = 0.0     # рублей за 1000 входных токенов
    price_out: float = 0.0    # рублей за 1000 выходных

    @property
    def cost(self):
        return (self.tokens_in / 1000 * self.price_in +
                self.tokens_out / 1000 * self.price_out)

    def report(self):
        return (f"обращений: {self.calls}   "
                f"токенов: {self.tokens_in} вход / {self.tokens_out} выход   "
                f"стоимость: {self.cost:.4f} руб.")


def approx_tokens(text):
    """Грубая ОЦЕНКА числа токенов до вызова API.

    Это оценка, а не замер: точное число даёт токенизатор конкретной
    модели (см. урок 1.2). Нужна, чтобы прикинуть стоимость заранее.
    """
    return max(1, len(text) // 3)


class LLM:
    def __init__(self, price_in=0.0, price_out=0.0, max_retries=4, timeout=60):
        cfg = config.settings()
        self.offline = cfg["OFFLINE"]
        self.model = cfg["MODEL"]
        self.base_url = cfg["LLM_BASE_URL"]
        self.max_retries = max_retries
        self.usage = Usage(price_in=price_in, price_out=price_out)
        self._client = None
        if not self.offline:
            from openai import OpenAI
            self._client = OpenAI(base_url=self.base_url,
                                  api_key=cfg["LLM_API_KEY"],
                                  timeout=timeout)

    def _offline_answer(self, messages):
        """Детерминированный ответ: одинаковый запрос — одинаковый ответ."""
        text = " ".join(m["content"] for m in messages)
        h = hashlib.sha256(text.encode()).hexdigest()[:6]
        return (f"[автономный режим] Ответ-заглушка {h}. "
                f"Получено сообщений: {len(messages)}, символов: {len(text)}. "
                f"Подставьте ключ, чтобы обратиться к модели.")

    @staticmethod
    def _is_retryable(e):
        """Повторяем только то, что имеет шанс пройти со второго раза."""
        if type(e).__name__ in ("RateLimitError", "APITimeoutError",
                                "APIConnectionError", "InternalServerError",
                                "TimeoutError", "ConnectionError"):
            return True
        return getattr(e, "status_code", None) in (408, 429, 500, 502, 503, 504)

    def _with_retry(self, fn):
        """Экспоненциальная задержка со случайной добавкой."""
        for attempt in range(self.max_retries):
            try:
                return fn()
            except Exception as e:
                if not self._is_retryable(e) or attempt == self.max_retries - 1:
                    raise
                time.sleep(0.5 * (2 ** attempt) + random.uniform(0, 0.3))

    def ask(self, prompt, system=None, temperature=0.2, max_tokens=None):
        messages = ([{"role": "system", "content": system}] if system else []) + \
                   [{"role": "user", "content": prompt}]
        self.usage.calls += 1

        if self.offline:
            answer = self._offline_answer(messages)
            self.usage.tokens_in += approx_tokens(" ".join(m["content"] for m in messages))
            self.usage.tokens_out += approx_tokens(answer)
            return answer

        def call():
            kw = dict(model=self.model, messages=messages, temperature=temperature)
            if max_tokens:
                kw["max_tokens"] = max_tokens
            return self._client.chat.completions.create(**kw)

        resp = self._with_retry(call)
        u = getattr(resp, "usage", None)
        if u:
            self.usage.tokens_in += getattr(u, "prompt_tokens", 0)
            self.usage.tokens_out += getattr(u, "completion_tokens", 0)
        return resp.choices[0].message.content


In [ ]:
%%writefile llm-project/llmcourse/prompts.py
"""Промпт как часть программы. Урок 2.2.

Промпт хранится отдельно от кода вызова: у него есть имя, версия и явный
список параметров. Это позволяет менять формулировку, не трогая логику,
и видеть в журнале, какой версией промпта получен ответ.

Почему не str.format. Во-первых, .format умеет обращаться к атрибутам объекта
("{x.__class__}"), и на шаблоне, пришедшем извне, это дыра в безопасности.
Во-вторых, нам нужна подстановка и ничего больше — а всё лишнее в инструменте
рано или поздно кто-нибудь применит.

Правила шаблона те же, что в Python, чтобы знание переносилось:
    {name}  — параметр
    {{      — литеральная открывающая скобка
    }}      — литеральная закрывающая скобка
Одиночная } без пары — ошибка. Это не придирка: чаще всего она означает
незакрытый или неверно записанный параметр.
"""
import re
from dataclasses import dataclass

# Имя параметра — любой допустимый идентификатор, включая кириллицу: Python это
# разрешает. Но в примерах курса имена латинские — так принято, и так их видно
# в чужом коде без сюрпризов с раскладкой.
_NAME = re.compile(r"[^\W\d]\w*")


class PromptError(ValueError):
    """Ошибка в шаблоне промпта или в его заполнении."""


def parse(text):
    """Разбирает шаблон в список кусков: ("lit", строка) или ("field", имя)."""
    out, buf, i, n = [], [], 0, len(text)

    def flush():
        if buf:
            out.append(("lit", "".join(buf)))
            buf.clear()

    while i < n:
        ch = text[i]
        if ch == "{":
            if i + 1 < n and text[i + 1] == "{":
                buf.append("{"); i += 2; continue
            j = text.find("}", i + 1)
            if j == -1:
                raise PromptError("незакрытая « { » в шаблоне")
            name = text[i + 1:j]
            if not _NAME.fullmatch(name):
                raise PromptError(
                    f"недопустимое имя параметра: {{{name}}}. "
                    "Для литеральной скобки используйте {{ и }}")
            flush()
            out.append(("field", name))
            i = j + 1
            continue
        if ch == "}":
            if i + 1 < n and text[i + 1] == "}":
                buf.append("}"); i += 2; continue
            raise PromptError(
                "одиночная « } » в шаблоне. Для литеральной скобки пишите }}")
        buf.append(ch); i += 1
    flush()
    return out


@dataclass(frozen=True)
class Prompt:
    name: str
    version: str
    template: str
    system: str = ""

    @property
    def fields(self):
        """Имена параметров, которые нужно передать при заполнении."""
        parts = parse(self.template) + parse(self.system)
        return sorted({v for kind, v in parts if kind == "field"})

    def render(self, **values):
        """Заполняет шаблон. Молча ничего не проглатывает."""
        need, got = set(self.fields), set(values)
        if need - got:
            raise PromptError(
                f"{self.label()}: не переданы параметры {sorted(need - got)}")
        if got - need:
            raise PromptError(
                f"{self.label()}: лишние параметры {sorted(got - need)}. "
                f"Ожидались {self.fields}. Опечатка в имени — частая причина")

        def fill(text):
            return "".join(
                v if kind == "lit" else str(values[v]) for kind, v in parse(text))

        return {"system": fill(self.system), "user": fill(self.template)}

    def label(self):
        """Метка для журнала: по ней потом видно, чем получен ответ."""
        return f"{self.name}@{self.version}"


class Registry:
    """Все промпты проекта в одном месте."""

    def __init__(self):
        self._items = {}

    def add(self, prompt):
        old = self._items.get(prompt.name)
        if old is not None and old.version == prompt.version:
            raise PromptError(
                f"промпт {prompt.name} версии {prompt.version} уже зарегистрирован. "
                "Меняете формулировку — поднимите версию")
        self._items[prompt.name] = prompt
        return prompt

    def get(self, name):
        if name not in self._items:
            raise PromptError(f"промпт {name} не найден. Есть: {sorted(self._items)}")
        return self._items[name]

    def names(self):
        return sorted(self._items)


In [ ]:
import importlib
importlib.invalidate_caches()
from llmcourse import config, client, prompts
for m in (config, client, prompts):
    importlib.reload(m)

print(config.describe())
print("\nКаркас уроков 2.1-2.2 на месте.")

---
## Шаг 1. Почему разбор — отдельная работа

Кажется, что всё просто: попросили JSON — разобрали `json.loads`. На практике на вход разбора приходит вот такое.

| Что приходит | Откуда берётся |
|---|---|
| <code>```json ... ```</code> | Модель обучена оформлять код, и она оформляет |
| `Вот результат: {...}` | Вежливость, которую не отключить одной фразой в промпте |
| `{...} Надеюсь, помог!` | То же самое, но с другой стороны |
| `{"a": 1, "b": "текст обор` | Упёрлись в `max_tokens` — ответ обрезан на полуслове |
| `Извините, я не могу...` | Модель отказалась выполнять задачу |
| `{'a': 1}` | Одинарные кавычки. Это Python, но не JSON |

**Даже при строгой схеме** ответ приходит текстом: в документации GigaChat это прямо названо «JSON-объект, обёрнутый в строку». Разбирать всё равно придётся.

### Одна ловушка, на которой ломаются самодельные решения

Первая мысль — вырезать JSON регулярным выражением от `{` до `}`. Работает ровно до первого такого ответа:

```json
{"note": "закрывающая } внутри строки", "b": 2}
```

Регулярное выражение остановится на скобке внутри строки и вернёт обрывок. Поэтому в модуле — посимвольный сканер, который знает про строки и экранирование. Это тридцать строк кода, и они окупаются в первый же день.

*Статус ячейки: проверено запуском.*

In [ ]:
%%writefile llm-project/llmcourse/structured.py
"""Структурированный вывод. Урок 2.3.

Модель возвращает текст. Даже когда вы просите JSON и даже когда поставщик
поддерживает строгую схему, на вход разбора приходит строка — и в ней, кроме
самого JSON, регулярно оказывается лишнее:

    ```json ... ```      обрамление markdown
    Вот результат: {...}  вежливая преамбула
    {...} Надеюсь, помог  постамбула
    {"a": 1,              обрыв по max_tokens

Поэтому разбор должен быть устойчивым, а проверка — строгой. Модуль делает
ровно две вещи: достаёт JSON из шумного текста и проверяет его на соответствие
ожидаемой форме.
"""
import json
import re

FENCE = re.compile(r"```[a-zA-Z]*\s*\n?(.*?)```", re.S)


class StructureError(ValueError):
    """Ответ модели не удалось привести к ожидаемой форме."""


def strip_fences(text):
    """Снимает обрамление ```...```, если оно есть."""
    m = FENCE.search(text)
    return m.group(1) if m else text


def find_json(text):
    """Находит первый сбалансированный JSON-объект или массив.

    Сканер посимвольный, а не регулярное выражение: скобки встречаются внутри
    строк, и регулярное выражение на этом ломается. Пример, на котором ломаются
    почти все самодельные варианты:

        {"note": "закрывающая } внутри строки"}
    """
    src = strip_fences(text)
    start = None
    for i, ch in enumerate(src):
        if ch in "{[":
            start = i
            break
    if start is None:
        raise StructureError(
            "в ответе модели нет ни { ни [ — JSON отсутствует. "
            f"Начало ответа: {src[:80]!r}")

    opener = src[start]
    closer = "}" if opener == "{" else "]"
    depth, in_str, esc = 0, False, False

    for i in range(start, len(src)):
        ch = src[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == opener:
            depth += 1
        elif ch == closer:
            depth -= 1
            if depth == 0:
                return src[start:i + 1]

    raise StructureError(
        "JSON начался, но не закончился. Самая частая причина — ответ обрезан "
        "по max_tokens: проверьте finish_reason, при значении 'length' "
        "поднимите потолок")


def parse(text):
    """Достаёт JSON из ответа модели и разбирает его."""
    raw = find_json(text)
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        raise StructureError(
            f"JSON найден, но не разобран: {e.msg} (позиция {e.pos}). "
            f"Фрагмент: {raw[max(0, e.pos - 30):e.pos + 30]!r}")


# ── проверка формы ────────────────────────────────────────────────────
def check(data, required=(), types=None):
    """Проверяет разобранные данные на соответствие ожиданиям.

    required — имена обязательных полей;
    types    — словарь «поле: тип» для полей, тип которых важен.

    Возвращает список претензий. Пустой список означает, что всё в порядке.
    Список, а не первая ошибка: за один проход видно всё, что не так, —
    это заметно быстрее при отладке промпта.
    """
    problems = []
    if not isinstance(data, dict):
        return [f"ожидался объект, получен {type(data).__name__}"]
    for f in required:
        if f not in data:
            problems.append(f"нет обязательного поля {f!r}")
        elif data[f] is None:
            problems.append(f"поле {f!r} пустое (null)")
    for f, t in (types or {}).items():
        if f in data and data[f] is not None and not isinstance(data[f], t):
            problems.append(
                f"поле {f!r}: ожидался {t.__name__}, получен {type(data[f]).__name__}")
    return problems


def parse_checked(text, required=(), types=None):
    """Разбор с проверкой. Бросает StructureError со всеми претензиями сразу."""
    data = parse(text)
    problems = check(data, required, types)
    if problems:
        raise StructureError("ответ не соответствует ожиданиям: " + "; ".join(problems))
    return data


# ── цикл починки ──────────────────────────────────────────────────────
def ask_structured(ask, prompt, required=(), types=None, attempts=3, log=None):
    """Запрашивает структурированный ответ, при неудаче показывает модели её ошибку.

    ask — функция, принимающая текст и возвращающая ответ модели.

    Ограничение числа попыток обязательно. Без него неудачный промпт
    превращается в бесконечный платный цикл — ровно та ошибка, которую
    разбирали в уроке 2.1.
    """
    if attempts < 1:
        raise ValueError("attempts должно быть не меньше 1")
    log = log if log is not None else []
    text = prompt

    for n in range(1, attempts + 1):
        answer = ask(text)
        try:
            data = parse_checked(answer, required, types)
            log.append({"attempt": n, "ok": True})
            return data
        except StructureError as e:
            log.append({"attempt": n, "ok": False, "error": str(e)})
            if n == attempts:
                raise StructureError(
                    f"не удалось получить корректный ответ за {attempts} попыт(ки). "
                    f"Последняя ошибка: {e}")
            text = (
                f"{prompt}\n\n"
                f"Предыдущий ответ не подошёл: {e}\n"
                "Верни только JSON-объект, без пояснений и без обрамления markdown."
            )
    raise AssertionError("недостижимо")


---
## Шаг 2. Разбор на «грязных» ответах

Десять видов ответа, каждый из которых вы встретите. Все — записанные строки, ключ не нужен.

*Статус ячейки: проверено запуском.*

In [ ]:
import importlib
importlib.invalidate_caches()
import llmcourse.structured
importlib.reload(llmcourse.structured)
from llmcourse.structured import parse, check, parse_checked, ask_structured, StructureError

good = [
    ('чистый JSON',                 '{"a": 1}'),
    ('обрамление ```json',          '```json\n{"a": 1}\n```'),
    ('обрамление без языка',        '```\n{"a": 1}\n```'),
    ('преамбула',                   'Вот результат:\n{"a": 1}'),
    ('постамбула',                  '{"a": 1}\n\nНадеюсь, это помогло!'),
    ('преамбула + забор + постамбула', 'Конечно! Вот JSON:\n```json\n{"a": 1}\n```\nОбращайтесь.'),
]
for name, raw in good:
    assert parse(raw) == {"a": 1}, name
    print(f"  [ok] {name}")

tricky = '{"note": "закрывающая } внутри строки", "b": 2}'
assert parse(tricky)["b"] == 2
print("  [ok] закрывающая скобка внутри строки не обманывает сканер")

nested = '{"a": {"b": [1, 2, {"c": 3}]}}'
assert parse(nested)["a"]["b"][2]["c"] == 3
print("  [ok] вложенные объекты и массивы")

assert parse('[{"a": 1}, {"a": 2}]')[1]["a"] == 2
print("  [ok] массив на верхнем уровне")

print("\nТеперь то, что должно давать ПОНЯТНУЮ ошибку:\n")
bad = [
    ('обрыв по max_tokens', '{"a": 1, "b": "текст обор'),
    ('модель отказалась',   'Извините, я не могу выполнить эту задачу.'),
    ('одинарные кавычки',   "{'a': 1}"),
    ('висячая запятая',     '{"a": 1,}'),
]
for name, raw in bad:
    try:
        parse(raw)
        raise SystemExit("проверка провалена: " + name)
    except StructureError as e:
        print(f"  [ok] {name}:\n       {str(e)[:110]}")

print("\nРазбор устойчив.")

**Обратите внимание на текст ошибок.** Сообщение «обрезан по max_tokens: проверьте finish_reason» экономит слушателю полчаса — он сразу знает, куда смотреть. Сообщение «JSONDecodeError: Expecting value» не экономит ничего.

Это общее правило: текст ошибки пишется для того, кто будет её читать в три часа ночи.

---
## Шаг 3. Проверка формы

Разобрать JSON мало. Дальше надо убедиться, что в нём есть то, что вам нужно, и в нужном виде.

Наша проверка возвращает **список претензий, а не первую ошибку**. Причина практическая: при отладке промпта вы за один прогон видите всё, что не так, а не чините по одной штуке за раз.

*Статус ячейки: проверено запуском.*

In [ ]:
cases = [
    ("всё на месте",
     {"date": "01.05", "event": "встреча"}, ("date", "event"), None),
    ("нет обязательного поля",
     {"event": "встреча"}, ("date", "event"), None),
    ("поле есть, но пустое",
     {"date": None, "event": "встреча"}, ("date", "event"), None),
    ("неверный тип",
     {"date": "01.05", "count": "три"}, ("date",), {"count": int}),
    ("сразу несколько проблем",
     {"count": "три"}, ("date", "event"), {"count": int}),
    ("вместо объекта массив",
     [1, 2], ("date",), None),
]

for name, data, required, types in cases:
    problems = check(data, required=required, types=types)
    mark = "ok " if not problems else "!! "
    print(f"[{mark}] {name}")
    for p in problems:
        print(f"        - {p}")

print()
d = parse_checked('```json\n{"date": "01.05", "event": "встреча"}\n```',
                  required=("date", "event"), types={"date": str})
print("parse_checked вернул:", d)

try:
    parse_checked('{"event": "встреча"}', required=("date", "event"))
except StructureError as e:
    print("\nparse_checked на плохом ответе:")
    print(" ", e)

---
## Шаг 4. Цикл починки

Если ответ не подошёл, можно вернуть модели её собственную ошибку и попросить ещё раз. Это работает заметно лучше, чем повторять тот же запрос: модель получает конкретную претензию, а не просто вторую попытку.

**Ограничение числа попыток обязательно.** Без него неудачный промпт превращается в бесконечный платный цикл — ровно та ошибка, которую разбирали в уроке 2.1. Здесь она опаснее: каждая попытка длиннее предыдущей, потому что к промпту добавляется текст ошибки.

*Статус ячейки: проверено запуском.*

In [ ]:
# Модель-заглушка, которая ошибается дважды, а на третий раз отвечает верно.
calls = []

def flaky(text):
    calls.append(text)
    if len(calls) == 1:
        return "Конечно! Дата — 01.05, событие — встреча."      # вообще без JSON
    if len(calls) == 2:
        return '```json\n{"event": "встреча"}\n```'             # нет поля date
    return '{"date": "01.05", "event": "встреча"}'               # наконец верно

log = []
data = ask_structured(flaky, "Извлеки дату и событие из текста",
                      required=("date", "event"), attempts=3, log=log)

print("Результат:", data)
print()
for r in log:
    status = "ок" if r["ok"] else "не подошёл"
    print(f"  попытка {r['attempt']}: {status}")
    if not r["ok"]:
        print(f"      {r['error'][:100]}")

print("\nЧто получила модель на третьей попытке:")
print("-" * 60)
print(calls[2])
print("-" * 60)

In [ ]:
# А теперь то, ради чего ограничение существует.
def never_works(text):
    return "Извините, не могу."

try:
    ask_structured(never_works, "дай json", required=("a",), attempts=2)
except StructureError as e:
    print("[ok]", e)

count = []
def counting(text):
    count.append(1)
    return "нет"

try:
    ask_structured(counting, "x", required=("a",), attempts=1)
except StructureError:
    pass
print(f"[ok] attempts=1 — ровно {len(count)} обращение, а не больше")

try:
    ask_structured(counting, "x", attempts=0)
except ValueError as e:
    print("[ok]", e)

ok_calls = []
def fine(text):
    ok_calls.append(1)
    return '{"a": 1}'
ask_structured(fine, "x", required=("a",), attempts=3)
print(f"[ok] при успехе с первой попытки повторов нет: {len(ok_calls)} обращение")

---
## Шаг 5. Что гарантирует схема, а что нет

Многие поставщики умеют принимать JSON-схему и обещают ответ, ей соответствующий. У GigaChat это поле `response_format`; разберём его устройство, потому что детали здесь важнее общих слов.

| Что передали | Что получите |
|---|---|
| `type: "text"` | Обычный текст. Других полей объект содержать не может |
| `type: "json_schema"` + схема **без** `required` | Произвольный JSON-объект. Схема фактически необязательна |
| схема с `required`, **без** `strict: true` | Ответ может содержать поля, не описанные в схеме |
| схема с `required` **и** `strict: true` | Строгое соответствие; ответ содержит все поля, включая необязательные |

Источник: раздел «Генерация структурированных данных» документации GigaChat, обновлён 20.07.2026. Ссылка — в материалах урока.

Практический вывод: **`strict: true` без массива `required` не даёт ничего.** Это ровно тот случай, когда настройка выглядит включённой, а работает не так, как вы думаете.

### И главное ограничение

Схема гарантирует **форму** ответа. Она ничего не гарантирует про **содержание**.

Ячейка ниже показывает это на примере из официальной документации — не выдуманном.

*Статус ячейки: проверено запуском на данных, взятых из документации.*

In [ ]:
# Схема и ответы взяты из документации GigaChat (раздел «Генерация
# структурированных данных», обновлён 20.07.2026). Мы их только разбираем.

SCHEMA_DESCRIPTION = "Дата в формате dd.mm"     # что просили в описании поля

# Два ответа на ОДИН И ТОТ ЖЕ запрос с ОДНОЙ И ТОЙ ЖЕ схемой,
# приведённые в двух разных примерах той же страницы документации:
answers = [
    ('пример обычного вызова',  '{\n    "date": "27 октября 2023",\n    "event": "рождение сына"\n}'),
    ('пример потоковой выдачи', '{\n    "date": "2023-10-27",\n    "event": "рождение сына"\n}'),
]

print("В описании поля просили:", SCHEMA_DESCRIPTION)
print()
for name, raw in answers:
    data = parse_checked(raw, required=("date", "event"), types={"date": str})
    print(f"  {name}:")
    print(f"    форма верна — поля на месте, тип строковый: {data}")
    print(f"    но date = {data['date']!r} — это не формат dd.mm")
    print()

print("Вывод, который стоит запомнить:")
print("  схема проверяет, что поле есть и что оно строка;")
print("  она не проверяет, что в этой строке написано.")
print()
print("Содержательные требования проверяются вашим кодом.")

import re
DDMM = re.compile(r"^\d{2}\.\d{2}$")

def check_date_format(data):
    if not DDMM.fullmatch(data.get("date", "")):
        return [f"поле 'date': ожидался формат dd.mm, получено {data.get('date')!r}"]
    return []

for name, raw in answers:
    data = parse(raw)
    print(f"  наша проверка, {name}: {check_date_format(data)}")

---
## Шаг 6. Живой запрос

Единственный шаг, которому нужен ключ. Просим модель извлечь данные и разбираем ответ нашим модулем.

Что смотреть: пришёл ли чистый JSON или с обрамлением, соблюдён ли формат поля, сколько попыток потребовалось.

*Статус ячейки: требует проверки на живом ключе. Автор материалов эту ячейку с настоящим ключом не запускал.*

In [ ]:
from llmcourse.prompts import Prompt

HAVE_KEY = not config.settings().get("OFFLINE", True)

extract = Prompt(
    name="extract_event", version="1.0",
    system="Ты извлекаешь данные из текста. Отвечай только JSON-объектом, без пояснений.",
    template=(
        "Извлеки из текста дату и событие.\n"
        "Ответ верни в виде JSON: {{\"date\": \"дд.мм\", \"event\": \"краткое название\"}}\n"
        "Если данных нет — верни null в соответствующем поле.\n\n"
        "Текст:\n{text}"
    ),
)

TEXT = "Отчётное совещание перенесли на 5 мая, всех предупредили заранее."

if not HAVE_KEY:
    print("Ключ не подключён — шаг пропускается.")
    print("Шаги 0-5 дают всё содержание урока; вернитесь сюда, когда получите доступ.")
else:
    llm = client.LLM()
    r = extract.render(text=TEXT)

    def ask(text):
        return llm.ask(text, system=r["system"], temperature=0.1, max_tokens=200)

    log = []
    try:
        data = ask_structured(ask, r["user"], required=("date", "event"),
                              types={"date": str, "event": str}, attempts=3, log=log)
        print("Разобрано:", data)
        print("Попыток потребовалось:", len(log))
        print("Проверка формата даты:", check_date_format(data) or "формат соблюдён")
    except StructureError as e:
        print("Не удалось получить структурированный ответ:")
        print(" ", e)
        print("\nЭто нормальный результат для практики: посмотрите журнал ниже")
        print("и подумайте, что поправить в промпте.")
    for row in log:
        print("  ", row)

---
## Задание

1. **Свой набор грязных ответов.** Добавьте в шаг 2 три случая, которых там нет. Подсказки: JSON внутри двух заборов подряд; ответ, начинающийся с переноса строки и пробелов; объект с русскими кавычками-ёлочками вместо прямых.

2. **Проверка содержания.** Напишите функцию, которая проверяет не форму, а смысл: дата не в будущем, название события не пустое и не длиннее 100 символов. Подключите её после `parse_checked`.

3. **Стоимость починки.** Дополните `ask_structured` подсчётом токенов по всем попыткам. Оцените, во сколько обходится промпт, которому нужны в среднем две попытки вместо одной.

### Повышенной сложности

4. Реализуйте вариант, в котором при первой неудаче промпт не дополняется текстом ошибки, а заменяется более подробной версией из реестра промптов (урок 2.2). Сравните, какой подход даёт результат с меньшим числом попыток.

5. Добавьте в проверку формы поддержку вложенных полей — например, `"author.name"`. Сейчас проверяются только поля верхнего уровня.

---
## Чек-лист

- [ ] Модуль `structured.py` создан, разбор проходит на всех десяти видах ответа
- [ ] Понимаю, почему JSON нельзя вырезать регулярным выражением
- [ ] Могу объяснить, чем `strict: true` без `required` отличается от `strict: true` с ним
- [ ] Проверка формы возвращает все претензии сразу, а не первую
- [ ] Цикл починки ограничен по числу попыток, и я знаю, зачем
- [ ] Могу объяснить на примере, почему схема не гарантирует содержание

## Частые проблемы

| Симптом | Причина и что делать |
|---|---|
| `StructureError: JSON начался, но не закончился` | Ответ обрезан. Посмотрите `finish_reason`: при значении `length` поднимайте `max_tokens` |
| `StructureError: нет ни { ни [` | Модель ответила текстом. Обычно это отказ или непонятая задача — прочитайте начало ответа, оно в тексте ошибки |
| `StructureError: JSON найден, но не разобран` | Синтаксис. Чаще всего одинарные кавычки или висячая запятая — посмотрите фрагмент в тексте ошибки |
| Схема задана, а поля всё равно не те | Проверьте, что в схеме есть массив `required` и что `strict` выставлен. Без `required` строгость не работает |
| Цикл починки не сходится | Ошибка в промпте, а не в модели. Посмотрите журнал: если все попытки дают одну и ту же претензию — промпт просит невозможное |
| Формат поля не соблюдается | Схема проверяет тип, а не содержимое. Проверяйте формат своим кодом |

## Что дальше

Урок 2.4 — **вызов функций и первый агент**. Модель научится не просто возвращать данные, а сообщать, какое действие нужно выполнить. Механика та же, что сегодня: схема, разбор, проверка. Разница в том, что по результату разбора программа что-то делает.